In [1]:
%load_ext autoreload
%autoreload 2

Testing for importing the datasets and making sure the script works properly in and of itself before integration.

Test the datasets are loading properly

In [2]:
import sys
from pathlib import Path
import pe_dataset

MAX_BYTES = int(1.024 * 10**6)
malicious_path = Path("../Data/Testing_set/malicious")
benign_path = Path("../Data/Testing_set/benign")

data = pe_dataset.build_dataset_from_dir(benign_dir=benign_path, malware_dir=malicious_path, max_bytes=MAX_BYTES)


Getting some output of the dataset to see what the tensors are returning

In [3]:
import random

i = 0

while i < 200:
    index = random.randint(0, len(data))
    print(data.__getitem__(idx=index))

    i+=1

(tensor([120, 156, 220,  ..., 224, 227,  17]), tensor(1))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 129,  21, 227]), tensor(1))
(tensor([ 77,  90, 144,  ..., 194, 253, 255]), tensor(0))
(tensor([120, 156, 236,  ..., 182, 201, 137]), tensor(1))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 237,  ..., 256, 256, 256]), tensor(1))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 156, 236,  ..., 256, 256, 256]), tensor(1))
(tensor([ 77,  90, 144,  ..., 256, 256, 256]), tensor(0))
(tensor([120, 

KeyboardInterrupt: 

# Split and load the training and testing datasets

In [ ]:
import config
import torch
import torch.optim as optim
import torch.nn as nn

from torch.utils.data import DataLoader
#from Models.MalConv2 import MalConv

# This exists within pe_dataset.py, but I have it here to make things easier to test: changes made to pe_dataset.py require a reload
def split(dataset: pe_dataset.PEBinaryDataset, test_split=0.3, rand_state=42):
    from sklearn.model_selection import train_test_split

    data_labels = dataset.labels

    train_paths, test_paths, train_labels, test_labels = train_test_split(
        dataset.file_paths, data_labels, test_size=test_split,
        random_state=rand_state, stratify=data_labels)

    return train_paths, test_paths, train_labels, test_labels

NUM_EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
LEARNING_RATE = 1e-3

(X_train_paths, X_test_paths, y_train, y_test) = split(data)

num_benign, num_mal = 0, 0

for label in y_train:
    if label == 0:
        num_benign+=1
    else:
        num_mal+=1

if num_mal == num_benign:
    print("Benign/Malicious split is perfectly balanced; this is not quite always the case.")
else:
    print(f"Benign/Malicious balance: 0->{num_benign}, 1->{num_mal}")

train_data = pe_dataset.PEBinaryDataset(file_paths=X_train_paths, labels=y_train, max_bytes=MAX_BYTES)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

test_data = pe_dataset.PEBinaryDataset(file_paths=X_test_paths, labels=y_test, max_bytes=MAX_BYTES)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

#model = MalConv.MalConv()   # Basic MalConv model here. Will likely neee to change this later down the line.
#model = model.to(DEVICE)

#criterion = nn.CrossEntropyLoss()   # TODO: This needs to load from the JSON config file!
#optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)    # TODO: This needs to load its parameters from the JSON config file!

# Checking to make sure the files loaded in properly. I have had errors before where at least a file doesn't load correctly and the pipeline crashes.
_, _, train_whats_happening = train_data.filter_readable_labels()
if len(train_whats_happening) != 0:
    print("Bad paths in training set:")
    for entry in train_whats_happening:
        print(f"Bad Paths: {entry[0]} | Label: {entry[1]} | Error: {entry[2]}")

_, _, test_whats_happening = test_data.filter_readable_labels()
if len(test_whats_happening) != 0:
    print("Bad paths in test dataset")
    for entry in test_whats_happening:
        print(f"Bad Paths: {entry[0]} | Label: {entry[1]} | Error: {entry[2]}")


Benign/Malicious split is perfectly balanced; this is not quite always the case.


## Simple training loop
This will be expanded upon in the actual loop for the pipeline proper. This is here as a baseline and to test data loading.

In [ ]:
class SimpleTrainingLoop():

    def __init__(self, model : MalConv.MalConv, train_loader : DataLoader, test_loader : DataLoader, criterion = nn.BCEWithLogitsLoss, optim, max_epochs):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optim
        self.max_epochs = max_epochs

    def _train_one_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in self.train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            self.optimizer.zero_grad()
            output = self.model(x)
            logits = output[0] if isinstance(output, tuple) else output

            loss = self.criterion(logits, y)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

        return total_loss / total_samples, total_correct / total_samples

    @torch.no_grad()
    def _run_eval(self):
        self.model.eval()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in self.test_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            output = self.model(x)
            logits = output[0] if isinstance(output, tuple) else output

            loss = self.criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

        return total_loss / total_samples, total_correct / total_samples


    def train(self):
        best_val_loss = float("inf")
        loss_prev = 99999999

        for epoch in range(self.max_epochs):
            train_loss, train_acc = self._train_one_epoch()
            test_loss, test_acc = self._run_eval()

            print(
                f"Epoch {epoch+1}/{self.max_epochs} | "
                f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
            )

            if train_loss > loss_prev:
                print(f"Previous loss ({loss_prev}) is less than current epoch loss ({loss}). Stopping early to circumvent overfitting")
                break

            if train_loss <= 0.0001:
                print("Loss has effectively reached 0 - there is no point to continue training.")
                break

        #test_loss, test_acc = run_eval(model, test_loader)
        #print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

loop = SimpleTrainingLoop(model=model, train_loader=train_loader, test_loader=test_loader, criterion=criterion, optim=optimizer, max_epochs=20)
loop.train()

KeyboardInterrupt: 

## For the the training loop I must:

1. Train the defender normally

2. Check if the warm up phase for the defender is exceeded

3. Discover what predictions the model made on a batch of malicious files

4. For the files that were correctly classified, place those into the attacker's input source

5. Train the attacker on the passed malicious binaries and run preturbations

6. Track whether or not each perturbation succeeds

7. Repeat 

In [ ]:
class Adversarial_Loop:
    def __init__(self, defender, adversary, warm_start_epochs,
                 train_loader: DataLoader, test_loader: DataLoader,
                 max_epochs: int = 10, criterion = nn.BCEWithLogitsLoss(), batch_size=32):
        self.defender = defender
        self.criterion = criterion
        self.adversary = adversary
        self.warm_start = warm_start_epochs
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.max_epochs = max_epochs
        self.defender_token = "def" # Just to help with disambiguation and magic strings
        self.adversary_token = "adv"

    def _train_defender(self, model: str, mode: str):
        self.defender.to(DEVICE)

        total_loss = 0.0
        total_acc  = 0.0
        total_samples = 0
        total_tp = 0
        total_tn = 0
        total_fp = 0
        total_fn = 0

        loader = None

        if mode.lower() == 'train':
            loader = self.train_loader
        elif mode.lower() == 'test':
            loader = self.test_loader
        else:
            raise IndexError(f"No such mode {mode.lower()}")

        for x, y in loader:
            self.defender.zero_grad()

            batch_perf = None
            if mode.lower() == 'train':
                batch_perf = self.defender.batch_eval(x, y, self.criterion)
            elif mode.lower() == 'test':
                batch_perf = self.defender.pred(x, y, self.criterion)
            else:
                raise IndexError("How did you get here?")

            batch_size = batch_perf["total"]

            total_loss += batch_perf["loss"] * batch_size
            total_acc  += batch_perf["accuracy"] * batch_size
            total_tp += batch_perf["tp"] 
            total_tn += batch_perf["tn"] 
            total_fp += batch_perf["fp"] 
            total_fn += batch_perf["fn"] 
            total_samples += batch_size

        avg_loss = total_loss / total_samples
        avg_acc  = total_acc / total_samples
        rate_tp  = total_tp / total_samples
        rate_fp  = total_fp / total_samples
        rate_tn  = total_tn / total_samples
        rate_fn  = total_fn / total_samples

        return {
            "loss": avg_loss, "accuracy": avg_acc, 
            "tp_rate": rate_tp, "fp_rate": rate_fp, "tn_rate": rate_tn, "fn_rate": rate_fn
        }

    def _train_adversary(self, model: str):
        self.adversary.to(DEVICE)
        self.defender.to(DEVICE)
        self.defender.eval()

        total_loss = 0.0
        total_success = 0.0
        total_batches = 0
        total_mal_probs = 0.0
        total_reward = 0.0

        for x, y in self.train_loader:
            y = y.view(-1)  # Figure out whether or not you need this dynamically

            mal_mask = y == 1
            if mal_mask.sum() == 0: # I probably need to change this!
                continue

            x_mal = x[mal_mask]

            batch_perf = self.adversary.batch_eval(x_mal=x_mal, edit_mask=None, defender=self.defender)

            total_loss      += batch_perf["loss"]
            total_success   += batch_perf["asr"]
            total_mal_probs += batch_perf["mean_defender_malware_prob"]
            total_reward    += batch_perf["reward"]
            total_batches += 1

        return {
            "loss":             total_loss      / max(total_batches, 1),
            "asr":              total_success   / max(total_batches, 1),
            "avg_malware_prob": total_mal_probs / max(total_batches, 1),
            "avg_reward":       total_reward    / max(total_batches, 1)
        }

    def _train_one_epoch(self, model: str, mode: str = "train"):  
        if model.lower() == self.defender_token:
            return self._train_defender(model=model, mode=mode)

        elif model.lower() == self.adversary_token:
            return self._train_adversary(model=model)

        else:   # Shouldn't get here, but sometimes weird shit happens
            raise IndexError(f"Adversarial_Loop._train_one_epoch got an unexpected token: {model.lower}")

    def train(self):
        #best_val_loss = float("inf")
        
        for epoch in range(self.max_epochs):
            def_train_performance = self._train_one_epoch(model=self.defender_token, mode='train')
            def_test_performance  = self._train_one_epoch(model=self.defender.token, mode='test')
            adv_train_performance = None
            
            if epoch >= self.warm_start:
                adv_train_performance = self._train_one_epoch(self.adversary_token)

            # Defender metrics
            def_loss = def_train_performance["loss"]
            def_acc  = def_train_performance["accuracy"]
            def_tp_rate = def_train_performance["tp_rate"]
            def_tn_rate = def_train_performance["tn_rate"]
            def_fp_rate = def_train_performance["fp_rate"]
            def_fn_rate = def_train_performance["fn_rate"]

            print(
                f"Epoch {epoch+1}/{self.max_epochs} | "
                f"\n"
                f"==== DEFENDER METRICS ===="
                f"\n"
                f"Train Loss: {def_loss:.4f}, Train Acc: {def_acc:.4f} | Train TP Rate: {def_tp_rate:.4f}, Train FP Rate: {def_fp_rate:.4f}, Train TN Rate: {def_tn_rate:.4f}, Train FN Rate: {def_fn_rate:.4f} |"
                f"\n"
                f"Test Loss: {0}, Test Acc: {0} | "
                f"\n"
            )

            if adv_train_performance is not None:

                adv_loss       = adv_train_performance["loss"]
                adv_asr        = adv_train_performance['asr']
                adv_avg_prob   = adv_train_performance["avg_malware_prob"]
                adv_avg_reward = adv_train_performance["avg_reward"]

                print(
                    f"==== ADVERSARY METRICS ===="
                    f"\n"
                    f"Loss: {adv_loss:.4f}, ASR: {adv_asr:.4f} |"
                    f"\n"
                    f"Average Malware Probability: {adv_avg_prob:.4f}, Average Reward: {adv_avg_reward:.4f} |"
                    f"\n"
                )


NameError: name 'nn' is not defined

In [34]:
import agents

defender = agents.Defender()
adversary = agents.CNNAttacker()
pipeline = Adversarial_Loop(adversary=adversary, defender=defender, max_epochs=10,
                            warm_start_epochs=3, train_loader=train_loader, test_loader=test_loader)

pipeline.train()

Epoch 1/10 | 
==== DEFENDER METRICS ====
Train Loss: 0.6284, Train Acc: 0.0830 | Train TP Rate: 0.1357, Train FP Rate: 0.0000, Train TN Rate: 0.5000, Train FN Rate: 0.3643 |
Test Loss: 0, Test Acc: 0 | 

Epoch 2/10 | 
==== DEFENDER METRICS ====
Train Loss: 0.3532, Train Acc: 0.1250 | Train TP Rate: 0.4714, Train FP Rate: 0.0000, Train TN Rate: 0.5000, Train FN Rate: 0.0286 |
Test Loss: 0, Test Acc: 0 | 

Epoch 3/10 | 
==== DEFENDER METRICS ====
Train Loss: 0.1903, Train Acc: 0.1250 | Train TP Rate: 0.4714, Train FP Rate: 0.0000, Train TN Rate: 0.5000, Train FN Rate: 0.0286 |
Test Loss: 0, Test Acc: 0 | 



TypeError: CNNAttacker.batch_eval() missing 1 required positional argument: 'edit_mask'

# PROJECT NOTES
## Data
- Benign data gathered from clean windows 10 ISO mounting and extraction of Windows 10 home.
- Malicious data was gathered from SOREL-20M dataset
    - The original idea for the EMBER2024 dataset fell through, as this dataset does NOT provide the raw binaries that I planned to utilize
    - The setup script needs to be revised and reworked to accomodate the new setup methodology
        - This also means that the README is also out of date, and needs to have its setup section rewritten
    - Families of malicious data were NOT tracked, thus any imbalance of malware family representation (ie: virus, trojan, backdoor, or worm) have not been tracked and, as far as I know, are not able to be tracked
        - This should not have an effect on defender robustness improvements, but that isn't something measured by this study


## The Demo
- For the demo, it might be best to have a sampled set of the full dataset, only about 2, 3, or 400 samples instead of the full 16K.
    - Make note of this for the presentation

- Models are restricted to MalConv and _____
    - This is intentional, at least for now
        - This is more effective than to introduce a new variable in unproven model architectures against a benchmark, rather than training the benchmark against its experminental self.

## Implementation

- BIG CHALLENGE: Getting the defender and attacker to play together
    - The attacker is pre-made knowing which regions it can actually modify
        - Maintains malicious functionality
        - Not the most effective evasion attempts, but this is a baseline showing that this can help improve defender capability
    - Struggle is that gradients don't map well for attacker without complex embedding solutions
        - Made input perturbation very complex for the attacker
        - Took a lot of time to get this right